In [1]:
import pandas as pd

# لاحظي حرف r قبل المسار
file_path = r"C:\Users\ev\Documents\ir_project-main\processed_documents.pkl"
df = pd.read_pickle(file_path)

# التأكد من نجاح التحميل
print("تم تحميل البيانات بنجاح!")
print(df.head())

تم تحميل البيانات بنجاح!
        doc_id                                      original_text  \
0  NCT00000102  Title: Congenital Adrenal Hyperplasia: Calcium...   
1  NCT00000104  Title: Does Lead Burden Alter Neuropsychologic...   
2  NCT00000105  Title: Vaccination With Tetanus and KLH to Ass...   
3  NCT00000106  Title: 41.8 Degree Centigrade Whole Body Hyper...   
4  NCT00000107  Title: Body Water Content in Cyanotic Congenit...   

                                        cleaned_text  
0  titl congenit adren hyperplasia calcium channe...  
1  titl lead burden alter neuropsycholog develop ...  
2  titl vaccin tetanu klh assess immun respons co...  
3  titl degre centigrad whole bodi hyperthermia t...  
4  titl bodi water content cyanot congenit heart ...  


In [2]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words("english"))

stemmer = PorterStemmer()

In [3]:
def preprocess_query(query):

    query = query.lower()

    query = re.sub(
        r'[^a-zA-Z\s]',
        '',
        query
    )

    tokens = word_tokenize(query)

    tokens = [
        word
        for word in tokens
        if word not in stop_words
    ]

    tokens = [
        stemmer.stem(word)
        for word in tokens
    ]

    return " ".join(tokens)

In [4]:
query = "Treatment for heart disease"

print(
    preprocess_query(query)
)

treatment heart diseas


In [5]:
df['cleaned_text'].head()

0    titl congenit adren hyperplasia calcium channe...
1    titl lead burden alter neuropsycholog develop ...
2    titl vaccin tetanu klh assess immun respons co...
3    titl degre centigrad whole bodi hyperthermia t...
4    titl bodi water content cyanot congenit heart ...
Name: cleaned_text, dtype: object

In [6]:
queries = [
    "Heart disease treatment",
    "Breast cancer therapy",
    "Diabetes clinical trial",
    "Congenital heart disease",
    "Calcium channel blockers",
    "Body water content",
    "Neuropsychological development",
    "Immune response vaccination",
    "Whole body hyperthermia",
    "Lead burden effects"
]

In [8]:
def query_service(query):

    processed_query = process_query(query)

    return {
        "original_query": query,
        "processed_query": processed_query
    }

In [10]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def process_query(query):

    query = query.lower()

    query = re.sub(r'[^a-zA-Z\s]', '', query)

    tokens = word_tokenize(query)

    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    tokens = [
        stemmer.stem(word)
        for word in tokens
    ]

    return " ".join(tokens)

In [11]:
for q in queries:
    print(query_service(q))
    print("-" * 50)

{'original_query': 'Heart disease treatment', 'processed_query': 'heart diseas treatment'}
--------------------------------------------------
{'original_query': 'Breast cancer therapy', 'processed_query': 'breast cancer therapi'}
--------------------------------------------------
{'original_query': 'Diabetes clinical trial', 'processed_query': 'diabet clinic trial'}
--------------------------------------------------
{'original_query': 'Congenital heart disease', 'processed_query': 'congenit heart diseas'}
--------------------------------------------------
{'original_query': 'Calcium channel blockers', 'processed_query': 'calcium channel blocker'}
--------------------------------------------------
{'original_query': 'Body water content', 'processed_query': 'bodi water content'}
--------------------------------------------------
{'original_query': 'Neuropsychological development', 'processed_query': 'neuropsycholog develop'}
--------------------------------------------------
{'original_q

In [12]:
import nltk
nltk.download('wordnet')

from nltk.corpus import wordnet

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ev\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [13]:
def get_synonyms(word):

    synonyms = set()

    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonyms.add(
                lemma.name().replace("_"," ")
            )

    return list(synonyms)

In [14]:
print(get_synonyms("heart")[:10])
print(get_synonyms("disease")[:10])

['inwardness', 'ticker', 'nerve', 'heart', 'bosom', 'sum', 'tenderness', 'center', 'heart and soul', 'warmness']
['disease']


In [16]:
def expand_query(query):

    words = query.lower().split()

    expanded = words.copy()

    for word in words:
        expanded.extend(
            get_synonyms(word)[:2]
        )

    return " ".join(set(expanded))

In [17]:
print(
    expand_query(
        "heart disease"
    )
)

inwardness disease heart ticker


In [18]:
from textblob import TextBlob

def suggest_query(query):

    return str(
        TextBlob(query).correct()
    )

In [19]:
print(
    suggest_query(
        "hart diseas"
    )
)

hart disease


In [21]:
def query_service_v3(query):

    corrected_query = str(TextBlob(query).correct())

    processed_query = process_query(corrected_query)

    expanded_query = expand_query(corrected_query)

    return {
        "original_query": query,
        "corrected_query": corrected_query,
        "processed_query": processed_query,
        "expanded_query": expanded_query
    }

In [22]:
result = query_service_v3("hart diseas treatment")
print(result)

{'original_query': 'hart diseas treatment', 'corrected_query': 'hart disease treatment', 'processed_query': 'hart diseas treatment', 'expanded_query': 'disease handling Lorenz Milton Hart intervention stag hart treatment'}


In [23]:
print(query_service_v3("heart disease treatment"))

{'original_query': 'heart disease treatment', 'corrected_query': 'heart disease treatment', 'processed_query': 'heart diseas treatment', 'expanded_query': 'inwardness disease ticker handling heart intervention treatment'}


In [24]:
def get_synonyms(word):

    synonyms = set()

    for syn in wordnet.synsets(word):

        for lemma in syn.lemmas():
            synonym = lemma.name().replace("_", " ")

            if synonym.lower() != word.lower():
                synonyms.add(synonym.lower())

        break

    return list(synonyms)

In [25]:
print(get_synonyms("heart"))
print(get_synonyms("cancer"))
print(get_synonyms("treatment"))

['bosom']
['malignant neoplastic disease']
['intervention']


In [33]:
medical_synonyms = {
    "heart": ["cardiac"],
    "cancer": ["tumor", "carcinoma"],
    "disease": ["illness", "disorder"],
    "treatment": ["therapy"],
    "vaccination": ["immunization"]
}

In [31]:
def expand_query(query):

    processed_query = process_query(query)
    words = processed_query.split()

    expanded = set(words)

    for w in words:
        if w in medical_synonyms:
            expanded.update(medical_synonyms[w])

    return list(expanded)

In [32]:
print(get_synonyms("heart"))
print(get_synonyms("disease"))
print(get_synonyms("cancer"))

['pump', 'ticker', 'spunk', 'nerve', 'bosom', 'mettle']
[]
['malignant neoplastic disease', 'crab']


In [34]:
def get_synonyms(word):

    synonyms = set()

    synsets = wordnet.synsets(word)[:5]

    for syn in synsets:
        for lemma in syn.lemmas():

            synonym = lemma.name().replace("_", " ").lower()

            if synonym == word.lower():
                continue

            if len(synonym.split()) > 2:
                continue

            if synonym.isalpha():
                synonyms.add(synonym)

    return list(synonyms)

In [36]:
medical_synonyms = {
    "heart": ["cardiac"],
    "cancer": ["tumor"],
    "treatment": ["therapy"],
    "vaccination": ["immunization"],
    "disease": ["illness"]
}

In [37]:
print(get_synonyms("heart"))
print(get_synonyms("disease"))
print(get_synonyms("cancer"))
print(get_synonyms("treatment"))
print(get_synonyms("vaccination"))

['inwardness', 'ticker', 'nerve', 'bosom', 'sum', 'center', 'centre', 'kernel', 'pump', 'meat', 'eye', 'core', 'substance', 'gist', 'pith', 'spunk', 'middle', 'mettle', 'essence', 'marrow', 'nub']
[]
['crab']
['intervention', 'discussion', 'discourse', 'handling']
['inoculation']


In [38]:
def get_synonyms(word):

    synonyms = set()

    synsets = wordnet.synsets(word)

    if not synsets:
        return []

    syn = synsets[0]   # أول معنى فقط (Improvement مهم)

    for lemma in syn.lemmas():

        synonym = lemma.name().replace("_", " ").lower()

        if synonym == word.lower():
            continue

        if not synonym.isalpha():
            continue

        if len(synonym) < 3:
            continue

        synonyms.add(synonym)

    return list(synonyms)

In [39]:
print(get_synonyms("heart"))
print(get_synonyms("disease"))
print(get_synonyms("cancer"))
print(get_synonyms("treatment"))

['bosom']
[]
[]
['intervention']


In [40]:
def get_synonyms(word):

    synonyms = set()

    synsets = wordnet.synsets(word)[:2]

    for syn in synsets:

        for lemma in syn.lemmas():

            synonym = lemma.name().replace("_", " ").lower()

            if synonym == word.lower():
                continue

            if not synonym.isalpha():
                continue

            if len(synonym) < 3:
                continue

            synonyms.add(synonym)

    return list(synonyms)

In [41]:
print(get_synonyms("heart"))
print(get_synonyms("disease"))
print(get_synonyms("cancer"))
print(get_synonyms("treatment"))

['pump', 'ticker', 'bosom']
[]
['crab']
['intervention', 'handling']


In [42]:
import re

def process_query_service(query, expand=True):

    query_clean = re.sub(r'[^a-z\s]', '', query.lower())
    words = query_clean.split()

    if not expand:
        return " ".join(words)

    medical_syns = {
        "heart": ["cardiac"],
        "disease": ["illness", "disorder"],
        "cancer": ["tumor", "carcinoma"],
        "diabetes": ["diabetic"],
        "treatment": ["therapy"],
        "vaccination": ["immunization"],
        "immune": ["immunity"],
        "clinical": ["medical"],
        "trial": ["study"]
    }

    expanded_words = list(words)

    for w in words:
        if w in medical_syns:
            for syn in medical_syns[w]:
                if syn not in expanded_words:
                    expanded_words.append(syn)

    return " ".join(expanded_words)

In [43]:
print(process_query_service("heart disease treatment"))
print(process_query_service("breast cancer therapy"))
print(process_query_service("diabetes clinical trial"))
print(process_query_service("immune response vaccination"))

heart disease treatment cardiac illness disorder therapy
breast cancer therapy tumor carcinoma
diabetes clinical trial diabetic medical study
immune response vaccination immunity immunization


In [44]:
!jupyter nbconvert --to script sallynew.ipynb

[NbConvertApp] Converting notebook sallynew.ipynb to script
[NbConvertApp] Writing 7753 bytes to sallynew.py


In [47]:
import pandas as pd
import os

# ضعي المسار الكامل هنا (تأكدي من وجود حرف r قبل المسار)
file_path = r"C:\Users\ev\Documents\ir_project-main\processed_documents.pkl"

# التحقق من وجود الملف قبل محاولة فتحه
if os.path.exists(file_path):
    df = pd.read_pickle(file_path)
    print("تم العثور على الملف وتحميله بنجاح!")
    print(df.head())
else:
    print(f"خطأ: الملف غير موجود في المسار: {file_path}")
    print("يرجى التأكد من أن الملف موجود في هذا المجلد بالتحديد.")

تم العثور على الملف وتحميله بنجاح!
        doc_id                                      original_text  \
0  NCT00000102  Title: Congenital Adrenal Hyperplasia: Calcium...   
1  NCT00000104  Title: Does Lead Burden Alter Neuropsychologic...   
2  NCT00000105  Title: Vaccination With Tetanus and KLH to Ass...   
3  NCT00000106  Title: 41.8 Degree Centigrade Whole Body Hyper...   
4  NCT00000107  Title: Body Water Content in Cyanotic Congenit...   

                                        cleaned_text  
0  titl congenit adren hyperplasia calcium channe...  
1  titl lead burden alter neuropsycholog develop ...  
2  titl vaccin tetanu klh assess immun respons co...  
3  titl degre centigrad whole bodi hyperthermia t...  
4  titl bodi water content cyanot congenit heart ...  


        doc_id                                      original_text  \
0  NCT00000102  Title: Congenital Adrenal Hyperplasia: Calcium...   
1  NCT00000104  Title: Does Lead Burden Alter Neuropsychologic...   
2  NCT00000105  Title: Vaccination With Tetanus and KLH to Ass...   
3  NCT00000106  Title: 41.8 Degree Centigrade Whole Body Hyper...   
4  NCT00000107  Title: Body Water Content in Cyanotic Congenit...   

                                        cleaned_text  
0  titl congenit adren hyperplasia calcium channe...  
1  titl lead burden alter neuropsycholog develop ...  
2  titl vaccin tetanu klh assess immun respons co...  
3  titl degre centigrad whole bodi hyperthermia t...  
4  titl bodi water content cyanot congenit heart ...  


In [51]:
import pandas as pd
import os

# 1. تحديد المسار بدقة باستخدام الـ raw string (r) لتجنب مشاكل الـ backslashes
target_dir = r"C:\Users\ev\Documents\ir_project-main"
file_name = "processed_documents.pkl"

# 2. دمج المسار والاسم برمجياً
full_path = os.path.join(target_dir, file_name)

# 3. التأكد من أن المجلد موجود، وإن لم يكن كذلك فسيقوم بإنشائه
if not os.path.exists(target_dir):
    os.makedirs(target_dir)

# 4. حفظ ملف الـ Pickle في المسار المحدد
df.to_pickle(full_path)

print(f"تم حفظ الملف بنجاح في المسار: {full_path}")

تم حفظ الملف بنجاح في المسار: C:\Users\ev\Documents\ir_project-main\processed_documents.pkl


In [52]:
import os

# ابحثي في المجلد الرئيسي للمستخدم
search_path = r"C:\Users\ev"
target_file = "processed_documents.pkl"

print("جاري البحث... قد يستغرق الأمر ثوانٍ قليلة.")

for root, dirs, files in os.walk(search_path):
    if target_file in files:
        print(f"وجدته! الملف موجود في: {os.path.join(root, target_file)}")
        break
else:
    print("عذراً، لم أجد الملف في مجلد المستخدم. هل أنتِ متأكدة من كتابة الاسم بشكل صحيح؟")

جاري البحث... قد يستغرق الأمر ثوانٍ قليلة.
وجدته! الملف موجود في: C:\Users\ev\Documents\ir_project-main\processed_documents.pkl


In [53]:
import os

target_dir = r"C:\Users\ev\Documents\ir_project-main"

# التأكد من أن المجلد موجود
if os.path.exists(target_dir):
    files = os.listdir(target_dir)
    print(f"محتويات المجلد {target_dir}:")
    for f in files:
        print(f" - {f}")
        
    if "processed_documents.pkl" in files:
        print("\n✅ تم العثور على الملف: processed_documents.pkl موجود داخل المجلد!")
    else:
        print("\n❌ الملف غير موجود في هذا المجلد. هل هو باسم مختلف؟")
else:
    print(f"المجلد {target_dir} غير موجود. تأكدي من المسار.")

محتويات المجلد C:\Users\ev\Documents\ir_project-main:
 - .git
 - .ipynb_checkpoints
 - anaconda_projects
 - app.py
 - Document_Clustering_Service.ipynb
 - embedding_service.py
 - ghazal.ipynb
 - ghazalsally.ipynb
 - ghazalsally.py
 - git
 - lama update 5.py
 - Lama5.ipynb
 - lama8+.ipynb
 - lama8+.py
 - lama_evaluation_rag_service.py
 - ltr_service.ipynb
 - ltr_service.py
 - ltr_service_bak.ipynb
 - maya.ipynb
 - my_old_files
 - new_maria.ipynb
 - parallel_hybrid_search.py
 - processed_documents.pkl
 - quora_cleaned_sample.csv
 - quora_embeddings.npy
 - retrieval_real_evaluation_report.csv
 - retrieval_service.py
 - sally3.ipynb
 - sally3.py
 - Untitled1.ipynb
 - Untitled14.ipynb
 - Untitled15.ipynb
 - Untitled19.ipynb
 - Untitled3.ipynb
 - __pycache__

✅ تم العثور على الملف: processed_documents.pkl موجود داخل المجلد!


In [54]:
import shutil
import os

# 1. المسار الحالي للملف (حيث وجدناه سابقاً)
source_path = r"C:\Users\ev\Documents\ir_project-main\processed_documents.pkl"

# 2. مسار الوجهة الجديد
target_folder = r"C:\Users\ev\Desktop\New folder"
target_path = os.path.join(target_folder, "processed_documents.pkl")

# 3. التأكد من وجود المجلد الوجهة، وإن لم يكن موجوداً نقوم بإنشائه
if not os.path.exists(target_folder):
    os.makedirs(target_folder)
    print(f"تم إنشاء المجلد: {target_folder}")

# 4. عملية النقل
try:
    shutil.move(source_path, target_path)
    print(f"✅ تم نقل الملف بنجاح إلى: {target_path}")
except Exception as e:
    print(f"حدث خطأ أثناء النقل: {e}")

✅ تم نقل الملف بنجاح إلى: C:\Users\ev\Desktop\New folder\processed_documents.pkl


In [55]:
import os
print("المجلد الذي يعمل فيه Jupyter الآن هو:", os.getcwd())

المجلد الذي يعمل فيه Jupyter الآن هو: C:\Users\ev\anaconda_projects\dd4c724e-f9ab-4a42-a7b6-82a4164cdb92


In [56]:
import pandas as pd
import os

# المسار الدقيق كما يظهر في جهازك
target_path = r"C:\Users\ev\Documents\ir_project-main\processed_documents.pkl"

# حفظ الـ DataFrame
df.to_pickle(target_path)

print(f"تم حفظ الملف بنجاح في: {target_path}")

تم حفظ الملف بنجاح في: C:\Users\ev\Documents\ir_project-main\processed_documents.pkl
